# 01 · Chạy thí nghiệm trên Colab

Notebook này **không chứa code mô hình**. Nó chỉ gọi `run.py`, vì mọi cấu hình nằm trong `configs/*.yaml`.

Thứ tự chạy: **0 → 1 → 2 → 3**. Sau đó chạy **4** (baseline), rồi **5** (ablation) theo từng ô.
Nếu Colab bị ngắt: chạy lại ô 1, rồi chạy lại đúng ô đang dở. Run đã xong sẽ được bỏ qua (`[skip ]`).

Nhớ chọn **Runtime → Change runtime type → GPU**.

## 0. Mount Drive và vào thư mục repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/drybean-1d-resnet'
%cd {PROJECT}
!git log --oneline | head -5

## 1. Dữ liệu

File `data/Dry_Bean_Dataset.xlsx` không nằm trên GitHub (`.gitignore` chặn). Ô này tìm file ở thư mục cũ `MyDrive/DryBean`
rồi chép sang. Nếu không tìm thấy, bạn tự chép file vào `MyDrive/drybean-1d-resnet/data/`.

In [ ]:
import os, glob, shutil, hashlib
DATA = 'data/Dry_Bean_Dataset.xlsx'
if not os.path.exists(DATA):
    found = glob.glob('/content/drive/MyDrive/DryBean/**/Dry_Bean_Dataset.xlsx', recursive=True)
    print('Tìm thấy:', found)
    if found:
        shutil.copy(found[0], DATA)
assert os.path.exists(DATA), 'Chưa có file dữ liệu trong data/'
print('md5 =', hashlib.md5(open(DATA, 'rb').read()).hexdigest())   # ghi số này vào README

## 2. GPU và phiên bản thư viện (lưu lại để người khác tái lập đúng môi trường)

In [ ]:
!nvidia-smi -L
import tensorflow as tf, sklearn, numpy, pandas
print('TF', tf.__version__, '| sklearn', sklearn.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
!pip freeze | grep -i -E "^(tensorflow|keras|numpy|pandas|scikit-learn|scipy|pyyaml|openpyxl|matplotlib)==" > results/env_colab.txt
!cat results/env_colab.txt

## 3. Smoke test (≈1 phút)

Chạy ResNet1 trên split 0 với 3 epoch, ghi vào `results_smoke/` (không đụng log thật).
Lần đầu chạy, `run.py` sẽ **tạo 5 file split** trong `splits/`.

Cần thấy: `split_k: train=9799 val=1089 test=2723`, và dòng `[done ]` có `params=2,044,679`.
Accuracy thấp là bình thường vì mới 3 epoch.

In [ ]:
!python -u run.py --smoke --models resnet1 --splits 0

## 4. Baseline: 3 model × 5 split = 15 run (≈1 giờ)

Mỗi run in một dòng `[done ]`. Theo dõi `params`, `epochs` (số epoch thực chạy), `best_ep`, `test_acc`.

In [ ]:
!python -u run.py

In [ ]:
!python aggregate.py --groups baseline

## 5. Ablation (mỗi ô 15 run)

Chạy lần lượt, ô sau sau khi ô trước xong. noES lâu nhất vì luôn chạy đủ 500 epoch.

In [ ]:
!python -u run.py --ablation configs/ablation_noBN.yaml

In [ ]:
!python -u run.py --ablation configs/ablation_ES_valacc.yaml

In [ ]:
!python -u run.py --ablation configs/ablation_noES.yaml

In [ ]:
!python aggregate.py

## 6. Đẩy kết quả lên GitHub

Cần secret `GH_TOKEN` trong Colab (biểu tượng chìa khoá ở thanh bên trái, bật *Notebook access*).
Ô này commit `splits/`, `results/` và notebook, rồi push. File `best.keras` và dữ liệu bị `.gitignore` chặn nên không lên.
**Lưu notebook (Ctrl+S) trước khi chạy ô này** để output được commit kèm.

In [ ]:
import subprocess
from google.colab import userdata
GH_USER = 'YOUR_GITHUB_USERNAME'          # <- sửa
REPO = 'drybean-1d-resnet'
MSG = 'Results: baseline + ablation'      # <- sửa cho đúng nội dung lần này

token = userdata.get('GH_TOKEN')
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***'))

sh('git config user.name "Dao Viet Trung" && git config user.email "daotrung2501@gmail.com"')
sh('git add splits results notebooks && git status --short | head -20')
sh(f'git commit -m "{MSG}"')
sh(f'git push https://{token}@github.com/{GH_USER}/{REPO}.git main')